In [ ]:
import pandas as pd
import numpy as np
import math
import os
from typing import List, Dict, Any, Literal
from pydantic import BaseModel

import time
from datetime import datetime

from datasets import load_dataset, load_from_disk
from sentence_transformers import SentenceTransformer

from dotenv import load_dotenv
from google import genai


load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

c:\Users\gongz\anaconda3\envs\gemini-rag-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load preprocessed dataset (skip pattern recognition)
agent_df = pd.read_csv("agent_df.csv")
agent_df = agent_df[['convo_id', 'Chunk_id', 'Text', 'action', 'action_prob']]
env_df = pd.read_csv("env_df.csv")
env_df = env_df[['convo_id', 'Chunk_id', 'Text', 'reaction', 'reaction_prob']]

In [ ]:
'''
ds = load_dataset(
    "gwenshap/sales-transcripts",
    data_dir="data/chunked_transcripts",
)
transcript = ds["train"]
transcript.save_to_disk("transcript_parquet")
'''

# Data Curation
## Load Training Data

In [2]:
# Convert Hugging Face dataset to pandas DataFrame
transcript = load_from_disk("transcript_parquet")
df = transcript.to_pandas()
df.drop("Embedding", axis=1, inplace=True)

# Display the DataFrame
print(f"DataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")

conversation_ids = {name: idx for idx, name in enumerate(df['Conversation'].unique())}
df['convo_id'] = df['Conversation'].map(conversation_ids)
print("number of conversations: ", len(conversation_ids))

# encode Speaker: Customer -> 0, Sales Rep -> 1
df['Speaker'] = df['Speaker'].str.lstrip('*')
df['Speaker'] = df['Speaker'].map({'Customer': 0, 'Sales Rep': 1})

DataFrame shape: (989, 4)

Column names: ['Conversation', 'Chunk_id', 'Speaker', 'Text']
number of conversations:  50


## Define State
In our application, state is defined to include:
(1) Chat history
(2) Action history
(2) Chat rounds
(3) Stage
(4) Customer Need/Intent
(5) Customer info (structural data)
(6) Sentiment

## Define Action
Greeting
Information Gathering
Recommendation
Objection Handling
Logistic
Follow-up Scheduling
Closing
Handoff

## Define Rewards
Final Conversion: +1
Customer End Conversation: -1
Customer express interest: +0.2
Customer agree to follow-up: +0.3
Customer objection: -0.3

## Prepare Training Data, Categorize Agent Response to Actions
1. Use Gemini to generate a preliminiary classification. 
2. Distillation

### API Distillation: Action Recognition

In [ ]:
class ActionDistillation(BaseModel):
    action: Literal["Greeting", "IG", "Recommendation", "OH", "FUS", "Closing", "Handoff", "Bait"]

def distill_action_with_prob(transcript:str):

    system_prompt = '''
    You are an expert sale representative and your task is to classify the customer service agent's response into exactly one of the following categories based on its primary intent:
    Greeting: Standard opening phrases used to welcome the customer or establish rapport.
    Information Gathering (IG): Actively asking questions to understand the customer’s specific needs, preferences, or background (e.g., "What size are you looking for?" or "How long have you had this issue?").
    Recommendation: Suggesting a specific product, service, or solution based on the customer's needs without offering a specific financial incentive.
    Objection Handling (OH): Directly addressing customer concerns, doubts, or complaints to pivot them back toward a positive outcome (e.g., "I understand your concern about the price, but our durability is unmatched.").
    Follow-up Scheduling (FUS): Coordinating a future time to reconnect, such as booking a demo, a call-back, or a maintenance appointment.
    Closing: Finalizing the transaction or ending the interaction once the primary goal is met (e.g., "Your order is confirmed," or "Have a great day!").
    Handoff: Explicitly transferring the customer to another department, a human agent, or a specialist (e.g., "Let me connect you with our billing department.").
    Bait: Offering a specific, time-sensitive, or conditional incentive (discounts, coupons, freebies) to motivate an immediate purchase or commitment.
'''

    config = genai.types.GenerateContentConfig(
        system_instruction = system_prompt,
        response_mime_type="application/json",
        response_schema=ActionDistillation,
        temperature=0.0,
        response_logprobs = True,
        logprobs = 1,
    )

    response = client.models.generate_content(
        model = "gemini-2.0-flash",
        contents = transcript,
        config = config,
    )

    result = response.parsed
    logprobs_list = response.candidates[0].logprobs_result.chosen_candidates

    action_probability = 0.0
    action_first_word = result.action.split()[0] if result.action else ""
    for entry in logprobs_list:
        if action_first_word and action_first_word in entry.token:
            action_probability = math.exp(entry.log_probability)
            break

    return result.action, action_probability

In [ ]:
agent_df = df[df['Speaker'] == 1].copy()
print(f"The number of agent action in the dataset: {len(agent_df)}")

batch_size = 50
cooldown_time = 30

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"agent_df_{timestamp}.csv"

batches = [agent_df[i:i + batch_size] for i in range(0, len(agent_df), batch_size)]

print(f"Starting processing: {len(batches)} batches of {batch_size} rows.")

for idx, batch in enumerate(batches):
    print(f"Processing batch {idx + 1}/{len(batches)}...")
    
    batch = batch.copy()
    batch[['action', 'action_prob']] = batch['Text'].apply(
        lambda x: pd.Series(distill_action_with_prob(x))
    )
    
    file_exists = os.path.isfile(output_file)
    batch.to_csv(output_file, mode='a', index=False, header=not file_exists)
    
    print(f"Batch {idx + 1} saved to {output_file}.")

    if idx < len(batches) - 1:
        print(f"Cooling down for {cooldown_time} seconds...")
        time.sleep(cooldown_time)


The number of agent action in the dataset: 511


KeyError: "['Embedding'] not found in axis"

In [ ]:
agent_df = pd.read_csv(output_file)

action_counts = agent_df['action'].value_counts()
print("Action Frequency Table")
print(action_counts)

agent_df.to_csv("agent_df.csv", index = False)


Action Frequency Table
action
IG                125
Closing           105
OH                 83
Greeting           75
FUS                70
Recommendation     41
Bait               12
Name: count, dtype: int64


#### API Distillation: Customer Reaction Recognition
Potential improvement: Feed chat history to the sentiment recognition too?

In [8]:
class CustomerReactionRecognition(BaseModel):
    reaction: Literal[
        "Interest", 
        "Objection", 
        "Conversion", 
        "FU", 
        "Neutral", 
        "Leave"
    ]

# --- 2. Define the Distillation Function ---
def distill_reaction_with_prob(transcript: str):
    """
    Classifies customer reaction and calculates the probability of the chosen category
    using logprobs from Gemini 2.0 Flash.
    """
    
    system_prompt = """
    You are an expert sales analyst and sentiment specialized in customer-agent interactions. 
    Your task is to analyze the customer's sentiment and intent in the provided transcript.

    ### Categories & Definitions:
    1. **Interest**: Customer asks clarifying questions, seeks details, or expresses positive curiosity.
    2. **Objection**: Customer expresses concerns about price, timing, or features but remains engaged in the conversation.
    3. **Conversion**: Customer explicitly agrees to the offer, signs up, or confirms they are ready to proceed.
    4. **Follow-Up (FU)**: Customer cannot commit now but requests a callback, email, or future contact.
    5. **Neutral**: Customer provides facts or answers questions without clear positive or negative sentiment.
    6. **Leave**: Customer explicitly rejects the offer, hangs up, or asks to be removed from the list.

    ### Instructions:
    - Analyze the transcript carefully, focusing on the customer's final stance.
    - Select exactly ONE category from the `CustomerReactionRecognition` schema.
    - Provide a brief justification for your choice to ensure internal consistency.
    """

    config = genai.types.GenerateContentConfig(
        system_instruction=system_prompt,
        response_mime_type="application/json",
        response_schema=CustomerReactionRecognition,
        temperature=0.0,
        response_logprobs=True,
        logprobs=1,
    )

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=transcript,
        config=config,
    )

    result = response.parsed
    # Access the logprobs from the first candidate
    logprobs_list = response.candidates[0].logprobs_result.chosen_candidates

    reaction_probability = 0.0
    # Match the first word of the reaction to find its logprob in the token list
    reaction_first_word = result.reaction.split()[0] if result.reaction else ""
    
    for entry in logprobs_list:
        if reaction_first_word and reaction_first_word in entry.token:
            reaction_probability = math.exp(entry.log_probability)
            break

    return result.reaction, reaction_probability

In [16]:
env_df = df[df['Speaker'] == 0].copy()
print(f"The number of customer response in the dataset: {len(env_df)}")

batch_size = 50
cooldown_time = 30

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"env_df_{timestamp}.csv"

batches = [env_df[i:i + batch_size] for i in range(0, len(env_df), batch_size)]

print(f"Starting processing: {len(batches)} batches of {batch_size} rows.")

for idx, batch in enumerate(batches):
    print(f"Processing batch {idx + 1}/{len(batches)}...")
    
    batch = batch.copy()
    batch[['reaction', 'reaction_prob']] = batch['Text'].apply(
        lambda x: pd.Series(distill_reaction_with_prob(x))
    )
    
    file_exists = os.path.isfile(output_file)
    batch.to_csv(output_file, mode='a', index=False, header=not file_exists)
    
    print(f"Batch {idx + 1} saved to {output_file}.")

    if idx < len(batches) - 1:
        print(f"Cooling down for {cooldown_time} seconds...")
        time.sleep(cooldown_time)


The number of customer response in the dataset: 478
Starting processing: 10 batches of 50 rows.
Processing batch 1/10...
Batch 1 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 2/10...
Batch 2 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 3/10...
Batch 3 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 4/10...
Batch 4 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 5/10...
Batch 5 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 6/10...
Batch 6 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 7/10...
Batch 7 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 8/10...
Batch 8 saved to env_df_20260127_194709.csv.
Cooling down for 30 seconds...
Processing batch 9/10...
Batch 9 saved to env_df_20260127_194709.csv.
Cooling down for 30 second

In [20]:
env_df = pd.read_csv(output_file)

reaction_counts = env_df['reaction'].value_counts()
print("Reaction Frequency Table")
print(reaction_counts)

print(f"Shape of env_df: {env_df.shape}")

Reaction Frequency Table
reaction
Interest      141
Objection     126
Neutral       125
Conversion     37
Leave          29
FU             20
Name: count, dtype: int64
Shape of env_df: (478, 7)


#### Process environment

Introduce `end_convo` as an indicator for the end of the trajectory.

If conversion occurs, truncate the trajectory.

Map `reaction` to `instant_reward` this will be used as `r` in the transition 


In [ ]:
from helper import vec_reward

env_df['end_convo'] = 0

# For each convo_id, find the row with the maximum Chunk_id and set end_convo=1 for that row
max_chunk = env_df.groupby('convo_id')['Chunk_id'].transform('max')
env_df['end_convo'] = (env_df['Chunk_id'] == max_chunk).astype(int)

def truncate_at_conversion(df):
    # Assumes df is sorted by Chunk_id ascending
    conv_idx = df.index[df['reaction'] == 'Conversion'].tolist()
    if conv_idx:
        first_conv_idx = conv_idx[0]
        # Keep through the first Conversion (inclusive)
        df = df.loc[:first_conv_idx]
        # Set 'end_convo' for the Conversion row
        df.loc[first_conv_idx, 'end_convo'] = 1
        # Set 'end_convo' for any previous rows (should be 0)
        if first_conv_idx > df.index[0]:
            df.loc[df.index[:-1], 'end_convo'] = 0
        return df
    else:
        return df

env_df = env_df.sort_values(['convo_id', 'Chunk_id']).reset_index(drop=True)
env_df = env_df.groupby('convo_id', group_keys=False).apply(truncate_at_conversion).reset_index(drop=True)
print(f"After truncation: {env_df.shape}")

env_df['round'] = env_df.groupby('convo_id')['Chunk_id'].cumcount() + 1

env_df['instant_reward'] = env_df['reaction'].apply(vec_reward)

After truncation: (414, 8)


C:\Users\gongz\AppData\Local\Temp\ipykernel_19908\2665564844.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  env_df = env_df.groupby('convo_id', group_keys=False).apply(truncate_at_conversion).reset_index(drop=True)


#### Helper: Load Chat History

In [26]:
speaker_map = {0: "Customer", 1: "Agent"}

def load_conversation_history(convo_id, chunk_id):
    # Select relevant rows
    rows = df[(df['convo_id'] == convo_id) & (df['Chunk_id'] < chunk_id)]
    # Sort by chunk_id
    rows = rows.sort_values('Chunk_id')
    # Format as "Speaker : Text"
    history = []
    for _, row in rows.iterrows():
        speaker = speaker_map[row['Speaker']]
        text = str(row['Text'])
        history.append(f"{speaker} : {text}")
    return "\n".join(history)

env_df['chat_history'] = env_df.apply(
    lambda row: load_conversation_history(row['convo_id'], row['Chunk_id']), 
    axis=1
)


## Embedding state

`state` is defined to be the collection of `chat_history` and structural data (currently: `round`, `cumulative_reward`)

Embedding model: `all-MiniLM-L6-v2` (light model, max_token: 256)

In [28]:
# 1. Initialize the model (it will download on first run ~80MB)
# This model is 384-dimensional
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded:{model}\n Model is running on {model.device} \n\n")

# set truncation side to left
model.tokenizer.truncation_side = 'left' 
model.max_seq_length = 256

def get_state_vector(chat_text, structural_data):
    embedding = model.encode(chat_text, convert_to_numpy=True)
    state = np.concatenate([embedding.flatten(), np.array(structural_data).flatten()])
    return state.astype(np.float32)

env_df['state_vector'] = env_df.apply(
    lambda row: get_state_vector(row['chat_history'], row[['round']]),
    axis=1
)

print("State vector are generated.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 584.19it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded:SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
 Model is running on cuda:0 


State vector are generated.


### Formating Into Transitions

In [31]:
# align action with the previous state
agent_actions = agent_df[['convo_id', 'Chunk_id', 'action']].copy()
agent_actions['Chunk_id'] -= 1 

env_for_transition = env_df[['convo_id', 'Chunk_id', 'state_vector', 'instant_reward', 'end_convo']].copy()
env_for_transition = env_for_transition.merge(agent_actions, on = ['convo_id', 'Chunk_id'], how = 'left')
# Penalty for 'Baiting' behavior
env_for_transition.loc[env_for_transition['action'] == 'Bait', 'instant_reward'] -= 0.5
# Calculate cumulative reward BEFORE state_0
env_for_transition['cumulative_reward'] = env_for_transition.groupby('convo_id')['instant_reward'].transform(
    lambda x: x.shift().cumsum().fillna(0)
)

# Append the reward to the state vector
env_for_transition['cumulative_reward'] = env_for_transition['cumulative_reward'].astype(float)
env_for_transition['state_vector'] = env_for_transition.apply(
    lambda row: np.append(row['state_vector'], row['cumulative_reward']), 
    axis=1
)
# keep s, r (instant_reward), a (action), d (end of trajectory)
env_for_transition = env_for_transition[['convo_id','Chunk_id','state_vector','instant_reward','action','end_convo']]

# s_0
transition_df = env_for_transition[env_for_transition['end_convo'] == 0].copy()
transition_df = transition_df.rename(columns={"state_vector": "state_0"})
transition_df = transition_df[['convo_id', 'Chunk_id', 'state_0', 'action', 'instant_reward']]

# Prepare s_1
post_transition_df = env_for_transition[env_for_transition['Chunk_id'] > 1].copy()
post_transition_df = post_transition_df.rename(columns = {"state_vector":"state_1"})
post_transition_df['Chunk_id'] -= 2
post_transition_df = post_transition_df[['convo_id','Chunk_id','state_1','end_convo']]

# Join s_1
transition_df = transition_df.merge(post_transition_df, on = ['convo_id','Chunk_id'], how = 'left')
transition_df.head()


,convo_id,Chunk_id,state_0,action,instant_reward,state_1,end_convo
0,0,1,"[-0.05248880013823509, 0.04998108744621277, 0....",IG,0.0,"[-0.0643913596868515, 0.03951812535524368, 0.0...",0
1,0,3,"[-0.0643913596868515, 0.03951812535524368, 0.0...",IG,-0.1,"[-0.10628529638051987, 0.10533787310123444, 0....",0
2,0,5,"[-0.10628529638051987, 0.10533787310123444, 0....",Recommendation,0.1,"[-0.063658706843853, 0.05465235188603401, 0.04...",0
3,0,7,"[-0.063658706843853, 0.05465235188603401, 0.04...",OH,-0.1,"[-0.08808907866477966, 0.06507407873868942, 0....",0
4,0,9,"[-0.08808907866477966, 0.06507407873868942, 0....",IG,-0.1,"[-0.036711279302835464, 0.04363728314638138, 0...",0


In [35]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"transition_df_{timestamp}.csv"
print(f"Effective Transition Count: {len(transition_df)}")
transition_df.to_csv(output_file)

Effective Transition Count: 364
